In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report,accuracy_score

import numpy as np
import pandas as pd

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [2]:
df=pd.read_csv(r'G:\Fast-api\ml-model-flask-api\project_02\data\insurance.csv')

In [3]:
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore,retired,High


In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
94,50,105.4,1.78,10.542289,False,Bangalore,government_job,Low
7,31,105.7,1.78,10.865821,True,Delhi,government_job,Medium
19,24,111.2,1.60,2.790000,True,Lucknow,student,High
42,23,69.9,1.79,2.600000,True,Mysore,student,Medium
12,42,95.2,1.78,17.580000,True,Chandigarh,freelancer,High


In [5]:
# here we copy the our data set because many thing we apply on the data set
df_feat=df.copy()

In [ ]:
# feature 1 : BMI
df_feat['bmi']=df_feat['weight'] / [df_feat['height']**2]
# we simply we add this feature in our data set

In [9]:
# feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    if age < 45:
        return "adult"
    if age < 60:
        return 'middle_age'
    return 'senior'

In [11]:
# with the upper function we divide the our data on that basis of that group
df_feat['age_group']=df_feat['age'].apply(age_group)
# now this column also going to add in the our data

In [12]:
# feature 3 : Lifestyle Risk
def lifestyle_risk(row):
    if row['smoker'] and row['bmi'] >30:
        return 'high'
    if row['smoker'] and row['bmi'] > 27:
        return 'medium'
    else:
        return 'low'
        

In [15]:
print(df_feat.columns.tolist())

['age', 'weight', 'height', 'income_lpa', 'smoker', 'city', 'occupation', 'insurance_premium_category', 'bmi', 'age_group']


In [16]:
print(df_feat[['smoker','bmi']].head())

   smoker                                                bmi
0   False  0     49.227482
1     35.772940
2     44.54193...
1   False  0     41.543393
1     30.189017
2     37.58923...
2   False  0     23.339908
1     16.960793
2     21.11838...
3    True  0     44.953978
1     32.667443
2     40.67519...
4    True  0     25.558843
1     18.573263
2     23.12611...


In [17]:
df_feat['bmi'] = df_feat['weight'] / (df_feat['height']/100)**2

In [19]:
df_feat['lifestyle_risk']=df_feat.apply(lifestyle_risk,axis=1)

In [21]:
print(df['city'].unique())

<ArrowStringArray>
[    'Jaipur',    'Chennai',     'Indore',     'Mumbai',       'Kota',
  'Hyderabad',      'Delhi', 'Chandigarh',       'Pune',    'Kolkata',
    'Lucknow',       'Gaya',  'Jalandhar',     'Mysore',  'Bangalore']
Length: 15, dtype: str


In [22]:
tier_2_cities=['Jaipur','Indore','Kota','Chandigarh','Lucknow','Gaya','Jalandhar','Mysore','Bangalore']
tier_1_cities=['Mumbai','Delhi','Bangalore','Chennai','Kolkata','Hyderabad','Pune']


In [23]:
# feature 4 : city tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    elif city in tier_3_cities:
        return 3
    

In [24]:
df_feat['city_tier']=df_feat['city'].apply(city_tier)

In [25]:
df_feat.drop(columns=['age','weight','smoker','city'])

,height,income_lpa,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
0,1.56,2.92000,retired,High,492274.819198,senior,low,2
1,1.83,34.28000,freelancer,Low,301890.172893,adult,low,1
2,1.64,36.64000,freelancer,Low,211183.819155,adult,low,2
3,1.55,3.34000,student,Medium,455359.001041,young,high,1
4,1.60,3.94000,retired,High,242968.750000,senior,high,2
...,...,...,...,...,...,...,...,...
95,1.57,19.64000,business_owner,Low,214207.472920,adult,low,2
96,1.54,34.01000,private_job,Low,479844.830494,adult,low,1
97,1.80,44.86000,freelancer,Low,187654.320988,middle_age,low,1
98,1.82,28.30000,business_owner,Low,305216.761261,adult,low,1


In [27]:
# now we divide the data set into x and y part
X=df_feat[['bmi','age_group','city_tier','income_lpa','occupation']]
y=df_feat['insurance_premium_category']

In [28]:
X,y

(              bmi   age_group  city_tier  income_lpa      occupation
 0   492274.819198      senior          2     2.92000         retired
 1   301890.172893       adult          1    34.28000      freelancer
 2   211183.819155       adult          2    36.64000      freelancer
 3   455359.001041       young          1     3.34000         student
 4   242968.750000      senior          2     3.94000         retired
 ..            ...         ...        ...         ...             ...
 95  214207.472920       adult          2    19.64000  business_owner
 96  479844.830494       adult          1    34.01000     private_job
 97  187654.320988  middle_age          1    44.86000      freelancer
 98  305216.761261       adult          1    28.30000  business_owner
 99  276887.781338       adult          1    28.16664  government_job
 
 [100 rows x 5 columns],
 0       High
 1        Low
 2        Low
 3     Medium
 4       High
        ...  
 95       Low
 96       Low
 97       Low
 98    

In [29]:
# now we differtiate the catgorical feature and numerical feature into the two parts
categorical_features=['age_grope','lifestyle_risk','occupation','city_tier']
numerical_features=['bmi','income_lpa']

In [55]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_cols = ['age', 'weight', 'height', 'income_lpa', 'bmi']
categorical_cols = ['city', 'occupation', 'smoker']  # tumhare actual categorical columns

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_cols),        # ✅ 3 values
    ('cat', OneHotEncoder(), categorical_cols)        # ✅ 3 values
])

In [56]:
preprocessor=ColumnTransformer(transformers=[(
    'cat',OneHotEncoder(),categorical_features),
    'num','passthrough',numerical_features
])

In [57]:
from sklearn.ensemble import RandomForestClassifier

# create the pipeline with the preprocessor and random forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [58]:
# now divide the data into two form
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [59]:
# now fit that data into the piplines
pipeline.fit(X_train, y_train)

ValueError: not enough values to unpack (expected 3, got 2)